# Esteira Geo — Processamento Interativo

Fluxo idêntico ao executado pelo watcher: **Bronze → Silver → Gold → PostGIS**

| Área | Caminho no bronze | Quem usa |
|------|-------------------|----------|
| Exploratório | `exploratorio/<use_case>/` | Jupyter (este notebook) |
| Automatizado | `automatizado/<use_case>/` | Watcher (pipeline automático) |

Arquivos no exploratório **não são movidos** para `processados/` — podem ser reutilizados livremente.

Os arquivos-fonte do pipeline estão disponíveis em `/app/pipeline_src/` no file browser do JupyterLab.

## 0. Configuração

In [ ]:
import os, sys
sys.path.insert(0, '/app/pipeline_src')  # código-fonte montado do host (editável via file browser)
sys.path.insert(0, '/app')               # módulos compilados da imagem (fallback)

# Altere aqui para processar outro use_case: enchentes_poa | enchentes_mg | enchentes_rj
# O Jupyter trabalha com bronze/exploratorio/ — o watcher usa bronze/automatizado/
os.environ['USE_CASE'] = 'enchentes_poa'

import importlib, config
importlib.reload(config)  # recarrega config com o USE_CASE atualizado

EXPLO_PREFIX = f'exploratorio/{config.USE_CASE}/'  # área de trabalho do Jupyter
AUTO_PREFIX  = f'automatizado/{config.USE_CASE}/'  # área monitorada pelo watcher

print(f"Use case    : {config.USE_CASE}")
print(f"Exploratório: s3://{config.AWS_S3_BRONZE_BUCKET}/{EXPLO_PREFIX}")
print(f"Automatizado: s3://{config.AWS_S3_BRONZE_BUCKET}/{AUTO_PREFIX}  (watcher)")
print(f"Silver      : s3://{config.AWS_S3_SILVER_BUCKET}/{config.S3_SILVER_PREFIX}")
print(f"Gold        : s3://{config.AWS_S3_GOLD_BUCKET}/{config.S3_GOLD_PREFIX}")
print(f"PostGIS     : {config.RDS_HOST}:{config.RDS_PORT}/{config.RDS_DATABASE}")
print(f"Fonte       : /app/pipeline_src/")

## 1. Inspecionar Bronze — Exploratório (S3)

In [ ]:
import boto3

s3 = boto3.client(
    's3',
    endpoint_url=config.AWS_ENDPOINT_URL,
    aws_access_key_id=config.AWS_ACCESS_KEY_ID,
    aws_secret_access_key=config.AWS_SECRET_ACCESS_KEY,
    region_name=config.AWS_S3_REGION_NAME,
)

resp = s3.list_objects_v2(Bucket=config.AWS_S3_BRONZE_BUCKET, Prefix=EXPLO_PREFIX)
bronze_files = [o['Key'] for o in resp.get('Contents', []) if not o['Key'].endswith('.keep')]
print(f"{len(bronze_files)} arquivo(s) em bronze/{EXPLO_PREFIX}:")
for f in bronze_files:
    print(f"  {f}")

## 2. Silver — Normalização

> Lê de `bronze/exploratorio/<use_case>/`. Os arquivos **não são movidos** para processados — o exploratório é não-destrutivo.

In [ ]:
from etl.silver_processor import process_silver

silver = process_silver(bronze_prefix=EXPLO_PREFIX, move_files=False)

if 'flooding' in silver:
    print(f"Áreas de enchente: {len(silver['flooding'])} registros")
    display(silver['flooding'].head())

if 'citizens' in silver:
    print(f"\nCidadãos: {len(silver['citizens'])} registros")
    display(silver['citizens'].head())

## 3. Gold — Batimento Geográfico (Spatial Join)

In [ ]:
from etl.gold_processor import process_gold, silver_ready

has_areas, has_citizens = silver_ready()
print(f"Silver pronto — áreas: {has_areas} | cidadãos: {has_citizens}")

if has_areas and has_citizens:
    affected, unaffected, all_citizens = process_gold()

    total = len(all_citizens)
    print(f"\nResultado do batimento:")
    print(f"  Atingidos    : {len(affected)} ({len(affected)/total*100:.1f}%)")
    print(f"  Não atingidos: {len(unaffected)}")
    print(f"  Total        : {total}")

    display(affected.head())
else:
    print("Silver incompleto — execute as células anteriores primeiro.")

## 4. PostGIS — Sincronização

In [ ]:
from etl.postgis_loader import load_to_postgis

load_to_postgis(sync_areas=has_areas, sync_citizens=(has_areas and has_citizens))
print("PostGIS sincronizado.")

## 5. Consultas no PostGIS

In [ ]:
import psycopg2, pandas as pd

conn = psycopg2.connect(
    host=config.RDS_HOST, port=config.RDS_PORT,
    dbname=config.RDS_DATABASE, user=config.RDS_USER, password=config.RDS_PASSWORD
)

use_case = config.USE_CASE

stats = pd.read_sql(f"""
    SELECT
        COUNT(*) AS total,
        SUM(CASE WHEN affected_by_flooding THEN 1 ELSE 0 END) AS atingidos,
        SUM(CASE WHEN NOT affected_by_flooding THEN 1 ELSE 0 END) AS nao_atingidos
    FROM {use_case}_citizens
""", conn)

display(stats)

sample = pd.read_sql(f"""
    SELECT citizen_id, name, affected_by_flooding, ST_AsText(geometry) AS geom
    FROM {use_case}_citizens
    WHERE affected_by_flooding = TRUE
    LIMIT 5
""", conn)

display(sample)
conn.close()

## 6. Visualização no Mapa (Leaflet via IFrame)

In [ ]:
from IPython.display import IFrame

# Abre o mapa do Flask (certifique-se que o serviço web está rodando na porta 5000)
IFrame(src=f'http://localhost:5000/map?use_case={config.USE_CASE}', width='100%', height=600)

## 7. Upload para o Exploratório

Faz upload de um arquivo local para `bronze/exploratorio/<use_case>/` e reprocessa interativamente.
O arquivo **não é movido** para `processados/` — pode ser reutilizado quantas vezes quiser.

In [ ]:
from etl.silver_processor import process_silver
from etl.gold_processor import process_gold, silver_ready
from etl.postgis_loader import load_to_postgis

# Ajuste o caminho para o arquivo que deseja explorar
local_file = '/data/bronze/automatizado/enchentes_poa/citizens_sample.csv'

# Destino: exploratorio/<use_case>/<filename>
s3_key = f"exploratorio/{config.USE_CASE}/{os.path.basename(local_file)}"
s3.upload_file(local_file, config.AWS_S3_BRONZE_BUCKET, s3_key)
print(f"Upload: s3://{config.AWS_S3_BRONZE_BUCKET}/{s3_key}")

# Reprocessar a partir do exploratório (não move arquivos)
silver = process_silver(bronze_prefix=EXPLO_PREFIX, move_files=False)
has_areas, has_citizens = silver_ready()
if has_areas and has_citizens:
    affected, unaffected, all_citizens = process_gold()
    load_to_postgis(sync_areas=True, sync_citizens=True)
    print(f"Reprocessado: {len(affected)} atingidos / {len(all_citizens)} total")